# 01: Stock Performance of Service Intermediaries

Are intermediaries underperforming the market and AI platforms as agentic AI rises?

Run `python analysis/financials/fetch_prices.py` first to refresh the data.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "analysis").exists():
    ROOT = ROOT.parent
FIN = ROOT / "analysis" / "financials"
FIG = ROOT / "analysis" / "figures"

prices = pd.read_csv(ROOT / "analysis/data/processed/stock_prices.csv", parse_dates=["date"])
tickers = pd.read_csv(FIN / "tickers.csv")
events = pd.read_csv(FIN / "events.csv", parse_dates=["date"])
wide = prices.pivot(index="date", columns="ticker", values="close")
wide.tail()

## Settings
Change these to focus the analysis.

In [ ]:
START = "2023-01-01"     # baseline date: every series = 100 here
SECTOR = "travel"         # travel | insurance | financial | subscriptions | ai_platform
COMPARE = ["SPY", "QQQ"]  # benchmarks to overlay

## Normalized performance (start = 100)

In [ ]:
def normalized(cols, start=START):
    w = wide.loc[start:, cols].dropna(how="all")
    return w / w.bfill().iloc[0] * 100

sector_tickers = tickers.loc[tickers.sector == SECTOR, "ticker"].tolist()
norm = normalized(sector_tickers + COMPARE)

fig, ax = plt.subplots(figsize=(11, 5.5))
for t in norm.columns:
    bench = {"SPY": dict(ls="--", color="black"), "QQQ": dict(ls=":", color="gray")}
    style = dict(lw=2.5, **bench.get(t, dict(ls="--", color="dimgray"))) if t in COMPARE else dict(lw=1.6)
    ax.plot(norm.index, norm[t], label=t, **style)
for _, e in events[events.date >= START].iterrows():
    ax.axvline(e.date, color="black", alpha=0.25, lw=1)
    ax.annotate(e.event, (e.date, ax.get_ylim()[1]), rotation=90, va="top", ha="right", fontsize=8, alpha=0.7)
ax.axhline(100, color="black", lw=0.8)
ax.set_title(f"{SECTOR.title()} intermediaries vs. market (indexed to 100 on {START})")
ax.set_ylabel("Indexed price")
ax.legend(ncol=4, frameon=False)
plt.tight_layout()
fig.savefig(FIG / f"stock_perf_{SECTOR}.png", dpi=200)
plt.show()

## Summary table: total return and excess return vs. SPY

In [ ]:
def summary(start=START):
    w = wide.loc[start:].dropna(axis=1, how="all")
    first, last = w.bfill().iloc[0], w.ffill().iloc[-1]
    ret = (last / first - 1) * 100
    out = tickers.set_index("ticker").join(ret.rename("total_return_%"), how="inner")
    out["excess_vs_SPY_%"] = out["total_return_%"] - ret["SPY"]
    return out[["company", "sector", "role", "total_return_%", "excess_vs_SPY_%"]].sort_values("excess_vs_SPY_%").round(1)

summary()

## Average by role
Do incumbent intermediaries as a group lag AI-native challengers and agent platforms?

In [ ]:
summary().groupby("role")[["total_return_%", "excess_vs_SPY_%"]].mean().round(1).sort_values("excess_vs_SPY_%")

## Event study: reaction on AI-agent announcement days
Fill in `events.csv` first. Excess return = stock's daily return minus SPY's on that day.

In [ ]:
rets = wide.pct_change() * 100
excess = rets.sub(rets["SPY"], axis=0)
intermediaries = tickers.loc[tickers.role == "incumbent intermediary", "ticker"].tolist()

rows = []
for _, e in events.iterrows():
    day = excess.index[excess.index.searchsorted(e.date)]  # next trading day if event on weekend
    rows.append({"event": e.event, "trading_day": day.date(), **excess.loc[day, intermediaries].round(2).to_dict()})
pd.DataFrame(rows) if rows else "Add events to analysis/financials/events.csv to run this." 

## Takeaways
- TODO: what does this show, and what does it *not* show? (See caveats in `analysis/financials/README.md`.)